**v0.12.1 JK data-safety hotfix:** The original author monthly CSV has months with `MP_median + CBI_median != pc1_hf`, often when high-frequency stock data are missing. The loader logs these dates and treats their shocks as missing rather than assuming zero. The factor/PCA/pricing methodology is unchanged.

# Equity Risk Premia — v0.12: Stability, priced factors and identified shocks

This standalone notebook **preserves the v0.11 published FF6/BAB/QMJ & residual-factor experiments** and adds three strictly distinct research blocks:

1. **Rolling residual & pricing-weighted PCs**, with each month traded using only previously observable portfolio returns.
2. **Common-sample Fama–MacBeth and two-step shrinkage GMM/SDF**, comparing holdout pricing using fixed formation-only portfolios, HAC and paired time-block bootstrap.
3. **Public high-frequency identified policy surprises:** Jarociński–Karadi monthly MP & CBI shock decomposition. Separate option to load genuine CPI-release surprises constructed from actual minus pre-announcement consensus.

**Important:** The RP method is a *residual/FF-conditional augmented-mean adaptation* of RP-PCA, **not** the complete Lettau–Pelger estimators and inference. FMB does not include Shanken corrections and GMM with shrinkage does not support standard J-test inference. Characteristic-sorted holdout portfolios can share constituent stocks. The high-frequency Fed shock identification relies on event-window/sign restrictions; no series of CPI surprises is inferred from FRED-level CPI. All results are research diagnostics, not deployed trading premia.

In [ ]:
import sys,base64,gzip,tarfile,io
from pathlib import Path
EMBEDDED="..."
RUNTIME=Path('/content/erp_v012_notebook_runtime') if Path('/content').exists() else Path.cwd()/'erp_v012_notebook_runtime'
RUNTIME.mkdir(parents=True,exist_ok=True)
with tarfile.open(fileobj=io.BytesIO(gzip.decompress(base64.b64decode(EMBEDDED))),mode='r:') as tf:
    tf.extractall(RUNTIME/'src',filter='data' if sys.version_info>=(3,12) else None)
sys.path.insert(0,str(RUNTIME/'src'))
import numpy as np,pandas as pd
from erp import (__version__,EquityData,analyze_equity_residuals,
    analyze_equity_factor_extensions,load_equity_fred_macro,
    analyze_equity_v012,load_jk_monetary_shocks,
    load_inflation_release_surprises,analyze_equity_shock_mapping,
    estimate_jk_shock_prices)
print('Embedded ERP source:', __version__)


## 1. Fixed discovery / test design

First 80 portfolios: 25 Size–B/M, 30 Industries, 25 Size–Momentum. Holdout 50: 25 Size–Profitability + 25 Size–Investment. **Constituent stocks can overlap** despite distinct portfolio definitions. Train through December 2005, OOS from January 2006. `GAMMAS` and `N_PCS` **must not be selected by validation performance**.


In [ ]:
MODEL='FF6_BAB_QMJ'
AS_OF='2026-10-31'
TRAIN_END='2005-12-31'
DISCOVERY=('SIZE_BM_25','INDUSTRY_30','SIZE_MOM_25')
VALIDATION=('SIZE_OP_25','SIZE_INV_25')
N_PCS=5
GAMMAS=(0.,12.,60.) # predeclared gamma=0 is the original PCA baseline
NULL_DRAWS=199
NULL_SEED=7331
MACRO_STATE_LAG_MONTHS=2
AQR_PATHS={} # original AQR workbook fallback if source blocked
FRED_LOCAL_PATH=None # OPTIONAL: original combined FRED CSV/ZIP from browser
CACHE=Path('erp_cache')
OUT=Path('erp_equity_v012_results');OUT.mkdir(parents=True,exist_ok=True)


## 2. Published Equity data, baseline v0.10 (unchanged)

The baseline remains untouched. Factor portfolios are fixed after formation, not continuously traded at zero cost. Neutrality against model factors holds in training, not necessarily OOS.

In [ ]:
data=EquityData.from_public_sources(cache_dir=str(CACHE),aqr=True,aqr_paths=AQR_PATHS)
base=analyze_equity_residuals(data,model=MODEL,discovery_universes=DISCOVERY,
    validation_universes=VALIDATION,train_end=TRAIN_END,as_of=AS_OF,n_pcs=N_PCS)
print('Training/OOS months:',base.metadata['formation_months'],base.metadata['oos_months'])
print('Original residual PC premia:');display(base.fmp_premia_tests.round(5))
print('Baseline mean-pricing errors / attribution:');display(base.holdout_metrics.round(5))


## 3. Test the existence of synchronous residual structure

Circular-shift each discovery-asset residual independently `NULL_DRAWS` times, preserving each asset's time path and autocorrelation while destroying synchronous cross-asset dependence. The rankwise eigenvalue null and conservative max-eigenvalue FWER comparison test common residual *covariance*, **not priced risks**. Compare principal subspaces between two formation-only halves to assess instability; this is not a formal confidence interval.

In [ ]:
ext=analyze_equity_factor_extensions(data,base,n_pcs=N_PCS,null_draws=NULL_DRAWS,
    null_seed=NULL_SEED,rp_gamma=GAMMAS,gx_k=N_PCS)
print('Residual eigenvalue null:');display(ext.eigen_null.round(6))
print('Formation subspace stability:');display(ext.subspace_stability.round(5))


## 4. Pricing-oriented residual factor challengers + SDF + three-pass

Formation covariance `C` is augmented by `gamma × a a′` using **training OLS alpha vector** `a` rather than zero-mean residual scores. This is an RP-PCA-*inspired* test: not the precise Lettau–Pelger procedure. Each direction is projected into net-dollar and FF/AQR-beta neutral portfolios using training betas. Mean and spanning-alpha tests use OOS months and HAC; Holm correction covers the whole predeclared gamma × PC family. Held-out asset means and SDF/GMM are also scored OOS.

Giglio–Xiu-style three passes: return PCA, cross-sectional lambda from training heldout assets, projection of **observed factors** onto latent factors. It is a point-estimate research diagnostic, not weak-factor robust inference.

In [ ]:
print('Predeclared pricing-oriented portfolios:');display(ext.pricing_oriented_premia.round(5))
print('Heldout mean-pricing errors, regularized SDF:');display(ext.pricing_oriented_sdf.round(5))
print('Three-pass observed FF/AQR premia diagnostics:');display(ext.three_pass_observed.round(5))
print('Three-pass validation errors:');display(ext.three_pass_diagnostics.round(5))


## 5. Download macro & cross-asset proxies automatically from FRED

`CPIAUCSL`: Consumer Price Index; `INDPRO`: Industrial Production; `DGS10` and `DGS2`: Treasury yields; `BAA10Y`: Moody's Baa–10Y Treasury spread; `VIXCLS`: CBOE volatility index. Monthly official observations are resampled without forward fill. Features: log YoY CPI/IP, 10Y–2Y slope, credit spread and log VIX. **First differences are statistical economic changes, not event-study identified macro surprises.**

The loader attempts a combined FRED download, then per-series downloads, then caches. If Colab cannot reach FRED, download the **original** combined CSV from `https://fred.stlouisfed.org/graph/fredgraph.csv?id=CPIAUCSL%2CINDPRO%2CDGS10%2CDGS2%2CBAA10Y%2CVIXCLS`, place its path in `FRED_LOCAL_PATH`, and re-run this cell. Do not fill unavailable values or substitute synthetic data.
**Colab file uploads:** if you have uploaded `fredgraph (5).zip` (or any `fredgraph*.zip` / `fredgraph*.csv`) in the `/content` Files panel, the next cell automatically discovers it and bypasses FRED downloads. You may still set `FRED_LOCAL_PATH` explicitly. Upload the file before executing this cell.


In [ ]:
# Auto-detect FRED files uploaded in Colab, even after previous cells ran.
# A manual FRED_LOCAL_PATH, if provided, takes precedence.
from pathlib import Path
fred_path = FRED_LOCAL_PATH
if fred_path is None:
    roots = [Path('/content'), Path.cwd()]
    candidates = []
    for root in roots:
        if root.exists():
            for pattern in ('fredgraph*.zip', 'fredgraph*.csv'):
                candidates.extend(p for p in root.glob(pattern) if p.is_file())
    candidates = sorted(set(candidates), key=lambda p: p.stat().st_mtime, reverse=True)
    if candidates:
        fred_path = str(candidates[0])
        print('Found uploaded FRED file:', fred_path)
    else:
        print('No local fredgraph ZIP/CSV found; attempting FRED or cached data.')

macro = None
try:
    macro = load_equity_fred_macro(cache_dir=CACHE/'macro', path=fred_path, as_of=AS_OF)
    print('FRED source:', macro.provenance['source'])
    print('First complete state month:', macro.provenance.get('first_complete_state_month'))
    display(macro.states.tail(6).round(4))
    display(macro.changes.tail(6).round(4))
except (ConnectionError, FileNotFoundError, ValueError) as exc:
    print('Economic-data acquisition unavailable:', exc)
    print('Factor discovery/pricing completed; economic decomposition NOT executed.')


## 6. FF and RPC economic factor mapping + conditional premia

**Contemporaneous interpretation:** regress FF/AQR and RPC returns on *monthly changes* of macro/cross-asset variables using formation dates, then evaluate the estimated mapping on OOS months. A high R² does **not** identify a causal macro risk premium.

**Predictability:** state variables are lagged by two complete calendar months and a Ridge forecast of factor expected returns is refitted expanding-window. Its OOS R² is compared with an expanding unconditional mean. Data are latest revised FRED historical observations, **not point-in-time vintages**. The optional one-step SDF numbers using economic changes are **proxy-priced moment diagnostics**, not validated prices of economically identified macro risks.

In [ ]:
macro_result=None
if macro is not None:
    macro_result=analyze_equity_residuals(data,model=MODEL,discovery_universes=DISCOVERY,
        validation_universes=VALIDATION,train_end=TRAIN_END,as_of=AS_OF,
        n_pcs=N_PCS,macro_shocks=macro.changes,state_predictors=macro.states,
        state_lag_months=MACRO_STATE_LAG_MONTHS)
    print('FF + RPC factor exposures to economic changes (NOT risk prices):')
    display(macro_result.macro_exposures.round(5))
    print('Cross-time stability / interpretation diagnostics:')
    display(macro_result.macro_diagnostics.round(5))
    print('Lagged macro-state factor return forecasting, OOS vs unconditional:')
    display(macro_result.state_scores.round(5))
    print('Tentative one-step macro proxy SDF — NOT identified shock risk prices:')
    display(macro_result.macro_sdf_pricing.round(5))
else:
    print('Macro mapping and forecast remain UNESTIMATED; no proxy or synthetic input used.')


## 7. v0.11 baseline exports and interpretation

All CSVs are separate. Conclude only when residual eigenvalue significance, OOS pricing, and OOS conditional prediction each survive their own tests. Neither residual R² nor FF6 alpha alone proves a new priced factor. No TAA factor rotation alpha is claimed.

In [ ]:
base.export(OUT/'equity_baseline_v010')
ext.export(OUT/'equity_extensions_v011')
if macro is not None:
    macro.export(OUT/'public_FRED_inputs')
    (OUT/'public_FRED_inputs'/'source_metadata.json').write_text(__import__('json').dumps(macro.provenance,indent=2))
    if macro_result is not None:macro_result.export(OUT/'equity_economic_mapping')
print('Saved to:',OUT.resolve())
for x in sorted(OUT.rglob('*.csv')):print(' -',x)


## 8. Rolling vs static: are the residual factors stable, and is pricing orientation useful?

**Freeze only at each formation date:** rolling windows of 120 months, rebalance annually. Each FMP is dollar-neutral and neutral to formation FF/AQR betas, using actual discovery-portfolio excess returns. Signs are aligned to the *previously known* factor direction. Annualized OOS premia are **not** automatically priced systematic risks. Prespecified gamma=0 vs gamma=12, first 3 PCs; no tuning on OOS results.

In [ ]:
ROLLING_WINDOW_MONTHS=120
ROLLING_REBALANCE_MONTHS=12
V012_N_PCS=3
V012_GAMMAS=(0.,12.)
pricing=analyze_equity_v012(data,base,formation_window=ROLLING_WINDOW_MONTHS,
    rebalance_months=ROLLING_REBALANCE_MONTHS,n_pcs=V012_N_PCS,
    gammas=V012_GAMMAS,bootstrap_draws=399)
print('Rolling PC premia:');display(pricing.rolling_premia.round(5))
print('First/last formation, turnover and FF neutrality:')
display(pricing.rolling_diagnostics.head(6).round(5))
display(pricing.rolling_diagnostics.tail(6).round(5))
pricing.export(OUT/'equity_v012_rolling_and_pricing')


## 9. FMB vs two-step GMM on precisely the same holdout assets and dates

Compare FF6/BAB/QMJ alone, FF+3 *plain residual PCs* (gamma=0), and FF+3 *augmented-mean PCs* (gamma=12). FMB formation betas and mean lambda estimates are fixed before the OOS period; two-step GMM uses shrinkage HAC weighting and the SDF pricing moments. Rolling FMP returns are **not** retrospectively inserted into the fixed-formation horse race. The bootstrap resamples **monthly loss differences** in common 12-month blocks; do not select gamma based on OOS results.

In [ ]:
print('Common-sample pricing: annualized mean RMSE, FMB vs two-step GMM')
display(pricing.pricing_comparison.round(6))
print('FMB training lambda estimates and HAC t-stats:')
display(pricing.fmb_lambdas.round(5))
print('Paired block-bootstrap differences vs original FF model:')
display(pricing.bootstrap_comparison.round(7))


## 10. Actual FOMC policy-information shocks — Jarociński–Karadi

The updated [author dataset](https://github.com/marekjarocinski/jkshocks_update_fed) contains monthly *MP_median* and *CBI_median* high-frequency announcement shocks. The two add to the announcement policy surprise. The panel is monthly since 1990, **not the same as changes in the level of the policy rate**; source currently ends March 2026, so months after that are missing, not zero. First attempt public GitHub, then cached copy, then a local original CSV (`shocks_fed_jk_m.csv`). This is independent of FRED macro download.

**Same-month fit** measures shock-factor comovement and is not a tradeable month-ahead predictor. The lag-1 experiment tests whether shocks observed during month t predict factor returns in month t+1. These are observational factor-response diagnostics, not yet an identified macro risk-premium estimator.
**Hotfix:** Original-source JK months with failed decomposition are excluded as missing (not zero) and recorded in shock provenance.


In [ ]:
from pathlib import Path
# Optional, if GitHub unavailable: upload author CSV and use its actual path here.
JK_LOCAL_PATH=None
if JK_LOCAL_PATH is None:
    candidates=sorted(Path('/content').glob('shocks_fed_jk_m*.csv')) if Path('/content').exists() else []
    if candidates:
        JK_LOCAL_PATH=str(candidates[-1]);print('Found uploaded JK file:',JK_LOCAL_PATH)
jk=None;shock_mapping=None
try:
    jk=load_jk_monetary_shocks(cache_dir=CACHE/'shock_data',path=JK_LOCAL_PATH,as_of=AS_OF)
    print('JK source:',jk.provenance['source'],'ends',jk.shocks.index.max())
    print('Source months excluded (unknown shocks, not zero):',jk.provenance.get('excluded_invalid_decomposition_months',[]))
    f,_=data.factors(MODEL)
    shock_mapping=analyze_equity_shock_mapping(f,pricing.rolling_returns,jk,
        train_end=TRAIN_END)
    if shock_mapping.empty:
        print('Insufficient common train/OOS months for JK analysis')
    else:
        display(shock_mapping.round(5))
    jk.export(OUT/'jk_announcement_shocks')
    shock_mapping.to_csv(OUT/'jk_ff_rpc_exposure_and_predictability.csv',index=False)
    # Pricing: nontraded-shock beta prices inferred cross-sectionally, FF controls.
    # This is a point-estimate diagnostic under high-frequency identifying assumptions.
    _,rf=data.factors(MODEL)
    holdout=pd.concat([data.excess_returns(k,rf).add_prefix(k+'__')
         for k in VALIDATION],axis=1)
    shock_price_summary,shock_price_lambda=estimate_jk_shock_prices(
         holdout,f,jk,train_end=TRAIN_END)
    print('JK nontraded-shock prices, heldout common sample (exploratory):')
    display(shock_price_summary.round(6))
    display(shock_price_lambda.loc[shock_price_lambda.factor.isin(
         ['MP_median','CBI_median'])].round(6))
    shock_price_summary.to_csv(OUT/'jk_common_sample_mean_pricing.csv',index=False)
    shock_price_lambda.to_csv(OUT/'jk_shock_fmb_lambda_hac.csv',index=False)
except (ConnectionError,ValueError,FileNotFoundError) as e:
    print('JK shocks unavailable: ',e)
    print('Download https://raw.githubusercontent.com/marekjarocinski/jkshocks_update_fed/main/shocks_fed_jk_m.csv')
    print('Upload the original CSV and re-run this cell; FRED levels are not used as a substitute.')


## 11. Inflation *release* surprises — optional separate input

There is no reliable automatic public consensus archive in this release. **Do not substitute** FRED CPI changes or Cleveland Fed expectations for verified announcement surprises. If you possess release-level data with `release_date,actual,consensus,consensus_source` (same rate units, pre-release consensus), load the CSV/ZIP. Results are separate from monetary-policy shocks. The parser cannot independently verify when a provider recorded its consensus.

In [ ]:
CPI_RELEASE_CSV_PATH=None
inflation_shocks=None
if CPI_RELEASE_CSV_PATH is not None:
    inflation_shocks=load_inflation_release_surprises(CPI_RELEASE_CSV_PATH,as_of=AS_OF)
    ff,_=data.factors(MODEL)
    cpi_exposures=analyze_equity_shock_mapping(ff,pricing.rolling_returns,
        inflation_shocks,train_end=TRAIN_END,min_oos=24)
    display(cpi_exposures.round(5))
    inflation_shocks.export(OUT/'inflation_release_surprises')
else:
    print('CPI announcement surprises NOT estimated: no verified actual/consensus release file')


## 12. Audit / next research decision

**Decision gates:** common residual risk, incremental OOS pricing after multiple-testing adjustment, pricing stability under rolling retraining, and pre-specified economic shock exposures/predictability must each be tested separately. Report uncertainty and any model selection as exploratory. This notebook does **not** claim published weak-factor robustness, identified inflation news when consensus is absent, or improved investable TAA value.

In [ ]:
print('ERP',__version__)
print('Core output:',OUT.resolve())
print('Rolling premia factors:',len(pricing.rolling_premia))
print('Pricing candidates x methods:',len(pricing.pricing_comparison))
print('FMB lambda rows:',len(pricing.fmb_lambdas))
print('GMM shrinkage status: diagnostics only; no standard J-test')
print('Policy shock mapping:', 'available' if shock_mapping is not None else 'not estimated')
print('CPI release shocks:', 'available' if inflation_shocks is not None else 'not estimated')
